# Unsupervised Learning – Cycling Patterns in Zurich with k-Means, Hierarchical Clustering and PCA

**Dataset:** Bike counts in Zurich 2024–2025: daily number of bikes at 16 automatic counting stations of the city of
Zurich, with weather and calendar information (Open Data Stadt Zürich). Files: `../00_Data/zurich_bike_counts_2024_2025.csv`
and `../00_Data/zurich_bike_stations.csv` (names and directions of the stations).

**Questions:** Which counting stations have a similar traffic pattern – commuter routes, inner-city routes, leisure
routes? And what distinguishes one day from another? There is **no target variable** – we look for groups and for a
compact description of the 16 stations.

**After this exercise you can ...**
- perform k-means iterations by hand and compute the within-cluster sum of squares,
- construct meaningful features for clustering,
- apply k-means (incl. scaling, elbow plot, silhouette analysis, several random starts) and interpret clusters,
- create and cut a dendrogram,
- use PCA to summarise many correlated features.

## Concept

The concepts behind this exercise (formulas, worked examples and additional explanations) are
explained in the notebook [clustering_pca_concept.ipynb](clustering_pca_concept.ipynb). Read it before you start.

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    adjusted_rand_score,
)
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Ignore warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: k-means by hand

*Use the notebook cell below as your calculator: type the formulas with the numbers in Python
(e.g. `(3 + 5) / 2` or `0.4 ** 2`) – no paper needed.*

Six points in two dimensions: A (1, 1), B (2, 1), C (4, 3), D (5, 4), E (1, 2), F (5, 5).
We use $k = 2$ with the initial centres $\mu_1 = A = (1, 1)$ and $\mu_2 = C = (4, 3)$.

a) Assign every point to its nearest centre (Euclidean distance).

b) Compute the new centres.

c) Repeat the assignment step. Has the algorithm converged?

d) Compute the WCSS of the final solution.

e) Check your results with `KMeans(n_clusters=2, init=np.array([[1, 1], [4, 3]]), n_init=1)`.

In [ ]:
# Exercise 1: six points and the initial centres A and C
points = pd.DataFrame(
    {'x1': [1, 2, 4, 5, 1, 5], 'x2': [1, 1, 3, 4, 2, 5]}, index=list('ABCDEF')
)
P = points.values
centres = np.array([[1.0, 1.0], [4.0, 3.0]])

# a) to c) Repeat: distances to both centres, assignment, new centres,
#    until the centres do not change any more
# Your code here

# d) Within-cluster sum of squares of the final clusters
# Your code here

# e) Check with scikit-learn
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 2: Describe each counting station by its traffic pattern

To cluster the 16 counting stations, we describe each station by a few **ratios** that characterise *who* cycles there
– independently of how busy the station is:

| feature | definition | high value means … |
|---|---|---|
| `weekend_ratio` | mean on weekends / mean on working days | leisure traffic |
| `summer_winter_ratio` | mean on working days in May–September / in November–February | fair-weather / leisure traffic |
| `rain_ratio` | mean on rainy working days (> 4 h rain) / on dry working days | insensitive to rain (commuting) |
| `school_holiday_ratio` | mean on working days during / outside school holidays | not dependent on school and university |
| `temp_corr` | correlation of the daily counts with the temperature | weather-dependent traffic |

a) Import the bike counts and the station list (`../00_Data/zurich_bike_stations.csv`). Compute the five features for each
station (the code for the first two is given).

b) Show the table sorted by `weekend_ratio` and describe the extremes.

In [ ]:
# a) Import the bike counts as bikes and the station list as stations
#    Files: '../00_Data/zurich_bike_counts_2024_2025.csv'
#           '../00_Data/zurich_bike_stations.csv'
# Your code here
S = stations['column'].tolist()  # the 16 station columns

# Masks (True/False for every day)
work = (bikes['weekend'] == 0) & (bikes['holiday'] == 0)
summer = bikes['month'].isin([5, 6, 7, 8, 9])
winter = bikes['month'].isin([11, 12, 1, 2])

# Features per counting station
st = pd.DataFrame(index=S)
st['name'] = stations.set_index('column')['name']
st['bikes_per_day'] = bikes[S].mean()
st['weekend_ratio'] = (
    bikes[bikes['weekend'] == 1][S].mean() / bikes[work][S].mean()
)
st['summer_winter_ratio'] = (
    bikes[work & summer][S].mean() / bikes[work & winter][S].mean()
)
# Your code here (rain_ratio, school_holiday_ratio, temp_corr)

# b) Table sorted by weekend_ratio
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 3: k-means – choosing k

a) Standardise the five ratio features (not `bikes_per_day` – we want to group stations by their *pattern*, not by
their size).

b) Fit `KMeans(n_clusters=k, n_init=10, random_state=42)` for $k = 1, \dots, 8$ and plot the WCSS (`inertia_`) against
$k$ (elbow plot). Additionally compute the silhouette score for $k = 2, \dots, 8$.

c) Which $k$ do you choose?

In [ ]:
# Exercise 3: the five ratio features
features = [
    'weekend_ratio',
    'summer_winter_ratio',
    'rain_ratio',
    'school_holiday_ratio',
    'temp_corr',
]

# a) Standardise
X = ___

# b) Elbow plot (k = 1..8) and silhouette score (k = 2..8)
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 4: Interpret the clusters

a) Fit k-means with $k = 3$ (`n_init=10, random_state=42`) and add the cluster labels to the station table.

b) Compute the mean of each feature per cluster and list the stations of each cluster. Give each cluster a meaningful
name.

c) Plot the average **weekly profile** (mean bikes per weekday, divided by the station's overall mean) of the three
clusters.

In [ ]:
# a) k-means with k = 3
km3 = ___
st['cluster'] = ___

# b) Mean of each feature per cluster and stations of each cluster
# Your code here

# c) Weekdays from Monday to Sunday
weekdays = [
    'Monday',
    'Tuesday',
    'Wednesday',
    'Thursday',
    'Friday',
    'Saturday',
    'Sunday',
]
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 5: Silhouette analysis

The mean silhouette score of Exercise 3 summarises a clustering in one number. A **silhouette analysis** looks at the
silhouette value $s_i = \frac{b_i - a_i}{\max(a_i, b_i)}$ of **every single station** ($a_i$ = mean distance to the
stations of its own cluster, $b_i$ = mean distance to the stations of the nearest other cluster):
$s_i$ close to 1 → well placed; close to 0 → between two clusters; negative → probably in the wrong cluster.

a) Compute the silhouette values of all stations with `silhouette_samples(X, labels)` for $k = 2, 3, 4$ and draw a
silhouette plot for each $k$: one horizontal bar per station, sorted within each cluster, with the station names and a
vertical line at the mean silhouette score (the plotting code is given).

b) Which stations are badly placed for $k = 3$? Compare the three plots. Does the silhouette analysis support $k = 3$?

In [ ]:
# Function: draws one bar per station with its silhouette value, by cluster
def silhouette_plot(X, labels, names):
    """Silhouette plot: one bar per station, sorted within clusters."""
    values = silhouette_samples(X, labels)
    table = pd.DataFrame({'name': names, 'cluster': labels, 'sil': values})
    table = table.sort_values(['cluster', 'sil'])
    colors = plt.colormaps['tab10'](table['cluster'])
    plt.barh(table['name'], table['sil'], color=colors)
    plt.axvline(values.mean(), color='black', linestyle='--')
    plt.xlabel('silhouette value')
    plt.title(
        'k = '
        + str(len(np.unique(labels)))
        + ': mean = '
        + str(round(values.mean(), 2))
    )


# a) Silhouette plots for k = 2, 3 and 4 (one plot per k)
names = st['name'].tolist()
plt.figure(figsize=(18, 6))
for i, k in enumerate([2, 3, 4]):
    labels_k = ___
    plt.subplot(1, 3, i + 1)
    silhouette_plot(X, labels_k, names)
plt.tight_layout()
plt.show()

# b) Silhouette value of each station for k = 3
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 6: Stability of k-means

k-means depends on the random initial centres.

a) Fit k-means with $k = 3$ and `n_init=1` for `random_state` = 0, 1, …, 19. Print the WCSS of each run and compare the
cluster assignment with the solution of Exercise 4 using the *adjusted Rand index* (ARI: 1 = identical partition,
independent of the cluster numbers; ≈ 0 = random agreement).

b) What do you conclude? Why is `n_init=10` a good idea?

In [ ]:
# Exercise 6: k-means with n_init=1 for random_state = 0, ..., 19;
#    WCSS and ARI compared with km3
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 7: Hierarchical clustering and dendrogram

a) Compute the hierarchical clustering of the standardised features with `linkage(X, method='ward')` and draw the
dendrogram with the station names as labels.

b) Cut the dendrogram into 3 clusters (`fcluster(Z, t=3, criterion='maxclust')`) and compare them with the k-means
clusters (`pd.crosstab`).

c) At which height would you cut the dendrogram? Which stations are merged last?

In [ ]:
# a) Ward linkage and dendrogram
Z = ___
# Your code here

# b) Cut into 3 clusters and compare with k-means
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 8: PCA – what makes days different?

Now we turn the perspective around: the **674 days** are the observations, the 16 stations are the features.

a) Standardise the daily counts of the 16 stations and fit a PCA. Plot the explained variance ratio of the first
components. How much variance does PC1 explain?

b) Show the loadings of PC1 and PC2 (`pca.components_`). Interpret both components.

c) Plot the days in the PC1–PC2 plane, coloured by weekend / working day. Which days have the highest and the lowest
PC2 scores?

In [ ]:
# a) PCA on the standardised counts of the 16 stations (674 days)
Z_days = StandardScaler().fit_transform(bikes[S])
pca = PCA()
scores = ___
# Your code here

# b) Loadings of PC1 and PC2 for each station
# Your code here

# c) Figure showing the days in the PC1-PC2 plane
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')